# Uplift Modeling in Marketing — Notebook 5: Confirmatory Evaluation in Sealed Test (S6)

Continuation of `03_Meta_Learners_PT.ipynb` and `04_Causal_Forest_Uplift_Trees_PT.ipynb`.
This notebook does not share the kernel of the previous ones — reload what
you need from `src/` and the persisted artifacts in `artifacts/s6/`.
Notebooks 03 and 04 remain frozen: no cell there was altered
in this round.

**Objective of Notebook 5 (S6):** confirmatory evaluation, once, of the
primary model pre-selected (`UpliftTree`) and three pre-specified comparators
(X+Tree, S+LightGBM, response-targeting baseline) in the sealed test —
the 20% partition of the dataset never used in any modeling decision until now.
All relevant decisions (primary model, comparators, metrics, interpretation
criterion, bootstrap protocol, models already trained and frozen) were taken
and recorded **before** the sealed test was opened — see `artifacts/s6/preregistration.json`
and `artifacts/s6/final_models.joblib`.

**The sealed test was opened only once in this authorized execution**,
after pre-registration and freezing of the models (see 6.1/6.2). Sections
6.4–6.6 record the confirmatory evaluation executed — primary model and
comparators scored once, metrics and bootstrap calculated, results persisted.
The sentinel
`SEALED_TEST_EVALUATED.json`, created at the end of this execution, prevents
any new execution of the evaluation.


## Contents

- [Section 6 — Confirmatory Testing in Sealed Test](#s6)
    - [6.1 Pre-Registration and Frozen Protocol](#s6-1)
    - [6.2 Frozen Final Models](#s6-2)
    - [6.3 Sealed Test](#s6-3)
    - [6.4 Confirmatory Evaluation](#s6-4)
    - [6.5 Primary-Hypothesis Result](#s6-5)
    - [6.6 Secondary Comparisons](#s6-6)
    - [6.7 Final Synthesis](#s6-7)

---

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import joblib
import pandas as pd

# Path bootstrap: allows `from src...` from the notebooks directory.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.i18n import make_lang
from src.viz import apply_plot_style

pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)

lang = make_lang('en')
apply_plot_style()

# S6 artifact paths; all were generated before opening the sealed test.
ARTIFACTS_S6_DIR = PROJECT_ROOT / 'artifacts' / 's6'
PREREG_PATH = ARTIFACTS_S6_DIR / 'preregistration.json'
FINAL_MODELS_PATH = ARTIFACTS_S6_DIR / 'final_models.joblib'
SEALED_TEST_SCORES_PATH = ARTIFACTS_S6_DIR / 'sealed_test_scores.parquet'
S6_RESULTS_PATH = ARTIFACTS_S6_DIR / 's6_results.json'
SENTINEL_PATH = ARTIFACTS_S6_DIR / 'SEALED_TEST_EVALUATED.json'


<a id='s6'></a>
<a id="s6"></a>

## Section 6 — Confirmatory Testing in Sealed Test

S4 and S5 evaluated seven candidates in total, always in `train_df`/`val_df` —
never in the sealed test. At the end of S5 (Notebook 04, section 5.8), this
development evaluation produced a recorded decision: `UpliftTree` as the primary model,
with X+Tree(depth=4), S+LightGBM(vanilla), and the response-targeting baseline
as pre-specified comparators. This section executes — when authorized — the only
confirmatory evaluation of the project: opening the sealed test once, scoring the
four frozen models (without any hyperparameter re-estimation) and testing the
primary hypothesis pre-registered with a bootstrap confidence interval calculated
before any result is observed.


<a id="s6-1"></a>

### 6.1 Pre-Registration and Frozen Protocol

Loads and prints `artifacts/s6/preregistration.json` — the complete specification registered before the test was opened: estimand, primary model, comparators, primary confirmatory hypothesis, metrics, bootstrap protocol, and the absolute guidelines for this round. The file's SHA-256 is computed and reported for integrity checking.


In [ ]:
prereg = json.loads(PREREG_PATH.read_text(encoding='utf-8'))
prereg_sha256 = hashlib.sha256(PREREG_PATH.read_bytes()).hexdigest()

assert prereg['status'] == 'preregistered_before_sealed_test'
assert prereg['created_before_test_unlock'] is True

labels = lang({'header': 'Pré-registro de S6 (artifacts/s6/preregistration.json)'})
print(f"{labels['header']}:\n")
print(f"  stage:                 {prereg['stage']}")
print(f"  status:                {prereg['status']}")
print(f"  estimand:              {prereg['estimand']}")
print(f"  primary_outcome:       {prereg['primary_outcome']}")
print(f"  primary_model:         {prereg['primary_model']}")
print(f"  comparators:           {prereg['comparators']}")
print(f"  excluded_from_s6:      {prereg['excluded_from_s6']['models']}")
print(f"  primary_hypothesis:    {prereg['primary_hypothesis']}")
print(f"  primary_metric:        {prereg['primary_metric']}")
print(f"  secondary_metrics:     {prereg['secondary_metrics']}")
print(f"  primary_comparison:    {prereg['primary_comparison']}")
print(f"  secondary_comparisons: {prereg['secondary_comparisons']}")
print(f"  bootstrap n_boot/seed: {prereg['bootstrap_protocol']['n_boot']} / {prereg['bootstrap_protocol']['seed']}")
print(f"  feature_cols:          {prereg['feature_cols']}")
print(f"  seed:                  {prereg['seed']}")
print(f"  created_before_test_unlock: {prereg['created_before_test_unlock']}")
print(f"\n  SHA-256 do arquivo: {prereg_sha256}")


<a id="s6-2"></a>

### 6.2 Frozen Final Models

Loads `artifacts/s6/final_models.joblib` — the four models trained once on `dev_df = train_df + val_df` (80% development, no sealed test data), with metadata saved along with the artifact. Verifies that the hash of the pre-registered model embedded in the metadata matches exactly with the hash calculated above in 6.1 — an integrity check between the two artifacts, not a re-opening of sensitive data.


In [ ]:
bundle = joblib.load(FINAL_MODELS_PATH)
models = bundle['models']
encoder = bundle['encoder']
metadata = bundle['metadata']

assert metadata['preregistration_sha256'] == prereg_sha256, (
    'The pre-registration hash embedded in final_models.joblib does not match the '
    'preregistration.json atual -- os modelos podem ter sido congelados sob '
    'uma especificação diferente da vigente.'
)

labels = lang({'header': 'Modelos finais congelados (artifacts/s6/final_models.joblib)'})
print(f"{labels['header']}:\n")
for name in metadata['model_names']:
    print(f"  {name:22s} -> {type(models[name]).__name__:28s} hyperparams={metadata['hyperparameters'][name]}")
print(f"\n  n_train (dev): {metadata['n_train']}")
print(f"  n_val   (dev): {metadata['n_val']}")
print(f"  n_dev total:   {metadata['n_dev']}  ({metadata['training_data']})")
print(f"  seed:          {metadata['seed']}")
print(f"  package_versions: {metadata['package_versions']}")
print(f"  created_before_test_unlock: {metadata['created_before_test_unlock']}")
print(f"  timestamp_utc: {metadata['timestamp_utc']}")
print('\n  Integridade preregistration_sha256 <-> preregistration.json: OK (hash bate)')


<a id="s6-3"></a>

### 6.3 Sealed Test

The sealed test can only be opened with explicit `unlock=True` in `load_sealed_test` (see `src/splits.py`). The cell below is an additional barrier, located after this notebook: while `UNLOCK_SEALED_TEST` remained `False`, execution would be interrupted before any cell that could access the test. `UNLOCK_SEALED_TEST` remained `False` throughout the preparation phase (pre-registration, model freezing, hash pre-flight). After explicit and specific authorization, the seal was changed to `True` and the cell below was executed once, opening the way for the confirmatory evaluation of 6.4. **`UNLOCK_SEALED_TEST = True` remains in the code as a historical record of this authorized execution** — it is not reverted to `False` retroactively. The post-execution state is protected by the sentinel `artifacts/s6/SEALED_TEST_EVALUATED.json` (see 6.4): any attempt to re-execute the confirmatory evaluation cell is automatically interrupted, without reopening the test nor recalculating anything.


In [ ]:
UNLOCK_SEALED_TEST = True  # Explicit authorization received -- S6 confirmatory evaluation

if not UNLOCK_SEALED_TEST:
    raise RuntimeError(
        "Teste selado permanece fechado. Pré-registro e modelos finais estão "
        "congelados (ver 6.1/6.2). Alterar UNLOCK_SEALED_TEST para True "
        "somente após autorização explícita, em uma rodada dedicada à "
        "avaliação confirmatória de S6."
    )


<a id="s6-4"></a>

### 6.4 Confirmatory Evaluation

**Executed only once after explicit authorization**, with `UNLOCK_SEALED_TEST = True` in 6.3. The cell first checked that `artifacts/s6/SEALED_TEST_EVALUATED.json` did not exist (protection against silent re-execution — without `force=True`); then loaded the four already frozen models from `final_models.joblib`, opened `test_df` only once via `load_sealed_test(df_pooled, unlock=True)`, generated the scores of the four models (no hyperparameter or model adjustment), calculated absolute Qini AUC/Uplift AUC/Uplift@30% of the four, and ran the specified paired bootstrap (`bootstrap_qini_comparison`, 2000 replicas, stratified by arm) for Qini AUC and the five deltas of 6.5/6.6. The results were persisted in `sealed_test_scores.parquet` + `s6_results.json`, and the sentinel `SEALED_TEST_EVALUATED.json` was created at the end, blocking any new execution of this cell.


In [ ]:
from datetime import datetime, timezone

from src.config import POOLED_TREATMENT_COL, PRIMARY_OUTCOME, TREATMENT_COL
from src.data import add_pooled_treatment, load_hillstrom
from src.evaluation import bootstrap_qini_comparison, evaluate_multiple_rankings
from src.learners import (
    encode_meta_learner_features, predict_propensity_score,
    predict_single_meta_learner, predict_uplift_tree_uplift,
)
from src.splits import dataset_fingerprint, load_sealed_test

if SENTINEL_PATH.exists():
    raise RuntimeError(
        f"{SENTINEL_PATH} already exists -- S6 has already been executed. This cell does not "
        "reopen the sealed test or silently recalculate metrics. "
        "Read s6_results.json for the result already produced; there is no option "
        "force=True para contornar esta proteção."
    )

df = load_hillstrom()
df_pooled = add_pooled_treatment(df)

test_df = load_sealed_test(df_pooled, unlock=True)  # the only sealed-test opening in this project

X_test = encode_meta_learner_features(test_df, encoder)
scores_test = {
    'UpliftTree': predict_uplift_tree_uplift(models['UpliftTree'], X_test),
    'X+Tree(depth=4)': predict_single_meta_learner('X', models['X+Tree(depth=4)'], X_test),
    'S+LightGBM(vanilla)': predict_single_meta_learner('S', models['S+LightGBM(vanilla)'], X_test),
    'Baseline (propensão)': predict_propensity_score(models['Baseline (propensão)'], test_df),
}

y_test = test_df[PRIMARY_OUTCOME].to_numpy(dtype=float)
treatment_test = test_df[POOLED_TREATMENT_COL].to_numpy()
arm_test = test_df[TREATMENT_COL].to_numpy()

metrics_table = evaluate_multiple_rankings(y_test, scores_test, treatment_test)
labels = lang({'header': 'Métricas absolutas no teste selado (única avaliação)'})
print(f"{labels['header']}:")
print(metrics_table.round(4))

deltas_spec = [
    ('UpliftTree', 'Baseline (propensão)'),
    ('UpliftTree', 'X+Tree(depth=4)'),
    ('UpliftTree', 'S+LightGBM(vanilla)'),
    ('X+Tree(depth=4)', 'Baseline (propensão)'),
    ('S+LightGBM(vanilla)', 'Baseline (propensão)'),
]
bootstrap_result = bootstrap_qini_comparison(
    y_test, treatment_test, arm_test, scores_test, deltas_spec,
    n_boot=prereg['bootstrap_protocol']['n_boot'], seed=prereg['bootstrap_protocol']['seed'],
)

scores_out = pd.DataFrame({
    'row_index': test_df.index.values,
    'score_uplift_tree': scores_test['UpliftTree'],
    'score_x_tree': scores_test['X+Tree(depth=4)'],
    'score_s_lgbm': scores_test['S+LightGBM(vanilla)'],
    'score_response_baseline': scores_test['Baseline (propensão)'],
})
scores_out.to_parquet(SEALED_TEST_SCORES_PATH, index=False)

final_models_sha256 = hashlib.sha256(FINAL_MODELS_PATH.read_bytes()).hexdigest()
results = {
    'metrics_absolute': metrics_table.round(6).to_dict(orient='index'),
    'bootstrap': bootstrap_result,
    'preregistration_sha256': prereg_sha256,
    'final_models_sha256': final_models_sha256,
    'dataset_fingerprint': dataset_fingerprint(),
    'n_test': int(len(test_df)),
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
}
S6_RESULTS_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')

SENTINEL_PATH.write_text(json.dumps({
    'evaluated': True,
    'timestamp_utc': results['timestamp_utc'],
    's6_results_path': str(S6_RESULTS_PATH),
}, indent=2), encoding='utf-8')

print(f"\nSalvos: {SEALED_TEST_SCORES_PATH.name}, {S6_RESULTS_PATH.name}, {SENTINEL_PATH.name}")


<a id="s6-5"></a>

### 6.5 Primary-Hypothesis Result

**Executed exactly once, in sequence after 6.4.** The interpretation rule already pre-registered in `preregistration.json` (`interpretation_rule_primary`) was automatically applied to the observed primary delta, ΔQini = Qini(UpliftTree) − Qini(response-targeting baseline), without changing any model or hyperparameter in this cell. See the result and automatic verdict in the output below, and the full reading in 6.7.


In [ ]:
delta_primary = bootstrap_result['deltas'][prereg['primary_comparison']]
print(f"ΔQini primário ({prereg['primary_comparison']}) = {delta_primary['point_estimate']:.4f}")
print(f"IC 95% bootstrap: [{delta_primary['ci_low']:.4f}, {delta_primary['ci_high']:.4f}]")

rule = prereg['interpretation_rule_primary']
if delta_primary['ci_low'] > 0:
    veredito = rule['ci_entirely_above_zero']
elif delta_primary['ci_high'] < 0:
    veredito = rule['ci_entirely_below_zero']
else:
    veredito = rule['ci_includes_zero']

print(f"\nVeredito (regra pré-registrada, fixada antes da abertura do teste):\n  {veredito}")


<a id="s6-6"></a>

### 6.6 Secondary Comparisons

**Executed exactly once, in sequence after 6.4.** The four pre-specified secondary deltas and the absolute values of the four candidates were shown, without selecting a new model based on them. See the output below and the full reading in 6.7.


In [ ]:
for key in prereg['secondary_comparisons']:
    d = bootstrap_result['deltas'][key]
    print(f"{key:45s} Δ={d['point_estimate']:+.4f}  IC95%=[{d['ci_low']:+.4f}, {d['ci_high']:+.4f}]")

labels = lang({'header': 'Valores absolutos dos quatro candidatos no teste selado'})
print(f"\n{labels['header']}:")
print(metrics_table.round(4))

print(
    "\nEssas comparações são pré-especificadas e reportadas por completude "
    "(inclusive para encerrar a pergunta de S4 sobre X+Tree/S+LightGBM vs. "
    "baseline), but they do not retrospectively replace the primary model or "
    "elegem um novo 'vencedor' com base no maior Qini absoluto observado aqui."
)


<a id="s6-7"></a>

### 6.7 Final Synthesis

**Primary confirmatory-hypothesis result.** On the sealed test (n=12,800, 20% of the dataset, never accessed before), ΔQini = Qini(UpliftTree) − Qini(response-targeting baseline) = -0.0088, with a paired bootstrap 95% CI stratified by the three original arms (2,000 replications) of [-0.0492, +0.0302]. Because the interval contains zero, the interpretation rule pre-registered in `preregistration.json` applies the verdict **"no confirmatory evidence of advantage"**: the primary hypothesis — that UpliftTree would produce a better incremental ranking than the response-targeting baseline — was not confirmed in this sealed sample. The point estimate is negative, but the CI is not entirely below zero, so there is also no confirmatory evidence that the baseline beats UpliftTree. Under this criterion, the result is inconclusive, not a statistically distinguishable defeat.

**Secondary results.** The four pre-specified deltas, all using the same paired bootstrap:

| Comparison | Δ | 95% CI |
|---|---|---|
| UpliftTree − X+Tree(depth=4) | -0.0381 | [-0.0682, -0.0097] |
| UpliftTree − S+LightGBM(vanilla) | -0.0086 | [-0.0412, +0.0220] |
| X+Tree(depth=4) − Response-targeting baseline | +0.0293 | [-0.0082, +0.0637] |
| S+LightGBM(vanilla) − Response-targeting baseline | -0.0003 | [-0.0295, +0.0309] |

Absolute Qini AUC values: UpliftTree 0.0089; X+Tree(depth=4) 0.0470; S+LightGBM(vanilla) 0.0175; response-targeting baseline 0.0177. The 95% bootstrap CI for the pre-specified secondary comparison UpliftTree − X+Tree(depth=4) excluded zero in favor of X+Tree ([-0.0682, -0.0097]). As a secondary comparison, this CI was not adjusted for multiplicity and does not replace the primary confirmatory hypothesis. Although X+Tree(depth=4) had the highest absolute Qini among the four candidates, its delta against the baseline includes zero, narrowly (lower CI -0.0082); therefore, this advantage is not distinguishable from zero under this criterion. The delta of S+LightGBM(vanilla) against the baseline is essentially null (-0.0003), closing the question left open in S4: on the sealed test, S+LightGBM does not show a distinguishable advantage over the response-targeting baseline. As pre-specified, none of these comparisons retroactively replaces the primary model or elects a new "winner" — UpliftTree remains the pre-registered primary model, regardless of the absolute Qini observed here for the other candidates.

**Whether the development pattern generalized.** In development (S4/S5, repeated holdout), UpliftTree had the highest mean Qini (0.0254) and the highest win rate (40%) among the six evaluated candidates, beating the baseline in 10 of 15 resamples — the main argument behind its choice as the primary model (Notebook 04, 5.8). **That pattern did not reproduce on the sealed test:** UpliftTree had the worst absolute Qini among the four candidates evaluated here (0.0089), below even its already weak result on the single fixed holdout of S5 (0.0105). That result had already been recorded in 5.7 as a "pattern of sensitivity to protocol/sample", and in this case it proved more informative about the sealed test than the repeated-protocol mean. On the other hand, X+Tree(depth=4) — which in development was in a "descriptive tie" with UpliftTree (mean Δ +0.0004, 8/15 splits) — had by far the best absolute performance on the sealed test, although without a bootstrap-CI distinction from the baseline. In short: for the pre-selected primary candidate, the sealed test does not confirm the pattern observed in the development protocol. That is itself a valid and informative result about the limits of the selection criterion used (partially overlapping repeated holdout, see 5.7), not an execution error in this evaluation.

**Limitations.**

- The sealed test is one single sample (n=12,800); the bootstrap CIs reflect resampling uncertainty within **this** sample, not variability across multiple possible sealed samples — unlike the development repeated holdout, which repeatedly resamples the same `train_df`.
- The four models were scored exactly once, with no retraining, as planned. With the data from this round, it is not possible to distinguish whether UpliftTree's weak test performance is noise specific to this 20% partition or a more general sign that the development selection criterion does not generalize well for this candidate.
- Five deltas were computed (1 primary + 4 secondary) without multiplicity correction for individual CIs — consistent with the pre-registration, which treats only the primary comparison as confirmatory and the others as descriptive.
- By design (pre-registered guardrails), this evaluation cannot be reopened or repeated. The sentinel `SEALED_TEST_EVALUATED.json` blocks any new execution of section 6.4 without `force=True` (which does not exist). Therefore, no independent replication is possible inside this project.
- This result must not be used to reselect a "winner" model or justify retrospective hyperparameter, protocol, or feature adjustments. None of those actions was taken in this round, and the primary hypothesis remains exactly as registered before the test was opened.

**State at the end of S6:** the primary confirmatory hypothesis (UpliftTree vs. response-targeting baseline) was not confirmed in this sealed sample. No new model was elected as UpliftTree's successor. The project closes the confirmatory evaluation phase with a negative result for the main hypothesis — recorded as such, without posterior reinterpretation.
